# task_13 Solution

In [1]:

from pathlib import Path
import pandas as pd

base = Path("../../tasks/task_13/data/")


In [2]:

tables = pd.read_csv(base / "tables.csv")
reservations = pd.read_csv(base / "reservations.csv")
orders = pd.read_csv(base / "orders.csv")


Q2: Among walk-ins that were seated on Friday through Sunday at 19:00 or later, which server had the highest median revenue per occupied seat-minute?

In [3]:
reservations["reservation_time"] = pd.to_datetime(reservations["reservation_time"])
reservations["seated_time"] = pd.to_datetime(reservations["seated_time"])
res = reservations.merge(tables[["table_id", "section", "capacity"]], on="table_id")

walkins = res[(res["channel"] == "walkin") & (res["no_show"] == 0)].copy()
walkins["day_name"] = walkins["reservation_time"].dt.day_name()
walkins["hour"] = walkins["reservation_time"].dt.hour
walkins = walkins[walkins["day_name"].isin(["Friday", "Saturday", "Sunday"]) & (walkins["hour"] >= 19)].copy()

analysis = walkins.merge(orders, on="reservation_id", how="inner")
analysis["revenue"] = analysis["food_total"] + analysis["drink_total"]
analysis["rev_per_seat_min"] = analysis["revenue"] / (analysis["party_size"] * analysis["duration_min"])

q2 = str(analysis.groupby("server")["rev_per_seat_min"].median().sort_values(ascending=False).index[0])
print(q2)

Dia


Q5: Among non-no-show reservations with party_size more than 3 on Friday through Sunday at 19:00 or later, which server generated the most total revenue?

In [4]:
rev = orders.merge(res[["reservation_id", "reservation_time", "party_size", "no_show"]], on="reservation_id")
rev = rev[(rev["no_show"] == 0) & (rev["party_size"] >= 4) & (rev["reservation_time"].dt.dayofweek >= 4) & (rev["reservation_time"].dt.hour >= 19)]
rev["total_rev"] = rev["food_total"] + rev["drink_total"]
q5 = str(rev.groupby("server")["total_rev"].sum().sort_values(ascending=False).index[0])
print(q5)


Bo
